# Topology Optimisation with EllPHi Gradients

This notebook demonstrates how to use `ellphi.grad` for gradient-based
optimisation of ellipsoid configurations. No dependencies beyond **ellphi**
and **scipy** are required.

**Sections**
1. Gradient-aware tangency (single pair)
2. Batch gradients and the VJP pattern (pairwise, with `scipy.optimize`)
3. [Placeholder] Persistence diagram optimisation

In [ ]:
import numpy as np
import ellphi
from ellphi import (
    tangency_grad, pdist_tangency_grad, pdist_tangency,
    coef_from_cov, coef_from_axes,
)

rng = np.random.default_rng(42)
print('ellphi version:', ellphi.__version__)

---
## 1. Gradient-aware tangency (single pair)

`tangency_grad(p, q)` returns a `TangencyGrad` dataclass with three fields:

| Field | Shape | Meaning |
|-------|-------|---------|
| `t` | scalar | Tangency distance (same as `tangency(p, q).t`) |
| `dt_dp` | `(m,)` | Gradient of `t` w.r.t. first-ellipsoid coefficients |
| `dt_dq` | `(m,)` | Gradient of `t` w.r.t. second-ellipsoid coefficients |

In [ ]:
# Build two 2-D ellipses from axes/angle representation
p = coef_from_axes(np.array([[0.0, 0.0]]), np.array([[3.0, 1.0]]), np.array([0.3]))[0]
q = coef_from_axes(np.array([[4.0, 1.0]]), np.array([[2.0, 0.5]]), np.array([-0.5]))[0]

g = tangency_grad(p, q)
print(f't          = {g.t:.6f}')
print(f'dt_dp      = {g.dt_dp}')
print(f'dt_dq      = {g.dt_dq}')

In [ ]:
# Quick finite-difference check for dt_dp[0]
from ellphi import tangency
h = 1e-6
p_plus = p.copy(); p_plus[0] += h
p_minus = p.copy(); p_minus[0] -= h
fd = (tangency(p_plus, q).t - tangency(p_minus, q).t) / (2 * h)
print(f'analytic dt_dp[0] = {g.dt_dp[0]:.8f}')
print(f'finite-diff       = {fd:.8f}')
print(f'relative error    = {abs(g.dt_dp[0] - fd) / abs(fd):.2e}')

---
## 2. Batch gradients and the VJP pattern

`pdist_tangency_grad(coefs)` returns:
- `dists`: condensed pairwise distance array (same as `pdist_tangency`)
- `vjp`: a pullback `grad_dists → grad_coefs` that accumulates upstream
  gradients into per-ellipsoid coefficient gradients

This VJP interface slots directly into `scipy.optimize.minimize(jac=True)`.

In [ ]:
from scipy.optimize import minimize

# Build a small cloud of N=6 random 2-D ellipses
N = 6
means0 = rng.uniform(-10, 10, (N, 2))
axes0  = rng.uniform(1.0, 4.0, (N, 2))
angles0 = rng.uniform(0, np.pi, N)

coefs0 = coef_from_axes(means0, axes0, angles0)
m = coefs0.shape[1]   # number of coefficients per ellipse

print(f'N={N} ellipses, m={m} coefficients each, {N*(N-1)//2} pairs')

In [ ]:
def objective_and_grad(x_flat):
    """Minimise the sum of pairwise tangency distances.
    
    Returns (f, g) as expected by scipy minimize(jac=True).
    """
    coefs = x_flat.reshape(N, m)
    dists, vjp = pdist_tangency_grad(coefs)

    # Objective: sum of all pairwise distances (want ellipses to be compact)
    f = float(np.sum(dists))

    # Gradient of f w.r.t. dists is the all-ones vector
    grad_dists = np.ones_like(dists)
    grad_coefs = vjp(grad_dists)

    return f, grad_coefs.ravel()


# Quick sanity: check the gradient at the initial point
f0, g0 = objective_and_grad(coefs0.ravel())
print(f'Initial objective: {f0:.4f},  |grad|={np.linalg.norm(g0):.4f}')

In [ ]:
result = minimize(
    objective_and_grad,
    coefs0.ravel(),
    method='L-BFGS-B',
    jac=True,
    options=dict(maxiter=50, ftol=1e-12, gtol=1e-6),
)
print(f'Converged: {result.success}  message: {result.message}')
print(f'Objective: {result.fun:.6f}  (initial: {f0:.6f})')

---
## 3. [Placeholder] Persistence diagram optimisation

This section shows *where* to insert TDA code to drive optimisation via
persistence diagrams. The gradient hookup via `vjp` is already in place from
Section 2 — only the TDA loss and its gradient need to be supplied.

```python
# ── USER-SUPPLIED TDA CODE ──────────────────────────────────────────────────
# import your_tda_library as tda
#
# def persistence_loss_and_grad(dists, n_ellipses):
#     """Compute TDA loss and its gradient w.r.t. pairwise distances.
#
#     Args:
#         dists:      condensed pairwise tangency-distance array (n_pairs,)
#         n_ellipses: number of ellipses N
#
#     Returns:
#         loss:       scalar
#         grad_dists: (n_pairs,) gradient of loss w.r.t. dists
#     """
#     dgm = tda.rips_persistence(dists, n_ellipses)
#     loss = tda.bottleneck_distance(dgm, target_dgm)
#     grad_dists = tda.bottleneck_gradient(dgm, target_dgm)
#     return loss, grad_dists
# ────────────────────────────────────────────────────────────────────────────
#
# def tda_objective_and_grad(x_flat):
#     coefs = x_flat.reshape(N, m)
#     dists, vjp = pdist_tangency_grad(coefs)
#
#     loss, grad_dists = persistence_loss_and_grad(dists, N)
#
#     # Chain rule: pull upstream gradient back through the VJP
#     grad_coefs = vjp(grad_dists)
#     return loss, grad_coefs.ravel()
#
# result = minimize(
#     tda_objective_and_grad,
#     coefs0.ravel(),
#     method='L-BFGS-B',
#     jac=True,
# )
```